# 📝 텍스트 임베딩 과제 LV3(통합) — 문의 군집·토픽·의미 검색

> 이 단원에서 배운 **임베딩·KMeans 군집·실루엣·대표 키워드·코사인 유사도**를 각각 하나의 작은 **프로그램**으로 완성하는 통합 과제입니다. 문제마다 여러 `### N단계` 셀로 나뉘어 있고, **각 단계 셀에 그 단계에서 할 일(요구 변수·기대 형태·주의)이 자립적으로** 적혀 있어요.

## 풀이 방법
1. 문제마다 **1단계에서 데이터를 불러와** 같은 변수(`docs`·`emb`·`labels` 등)를 뒷단계로 이어 씁니다.
2. 각 단계의 **답안 셀**(`# 여기에 코드를 작성하세요`)을 채우고, 아래 **자가채점 셀**(`# [자가채점]`)을 실행해 `✅ 통과!` 가 뜨면 성공이에요.
3. **그래프 단계**는 자가채점이 없습니다 — 위 **완성 그래프(정답)** 와 같은 모양으로 그리세요.
4. **인사이트 서술 단계**는 서술형입니다 — 정답 노트북의 모범 서술과 비교하세요.

- 데이터는 고객센터 **문의 20건**(`data/inquiries.csv`)입니다. 임베딩은 셀에서 직접 `emb_model.encode(...)` 로 만듭니다(모델은 처음 한 번만 내려받음).
- **KMeans 는 `random_state=0, n_init=10` 으로 고정**해야 채점 결과가 재현됩니다.

화이팅!

아래 두 셀을 먼저 실행해 이 단원에 필요한 라이브러리와 한국어 임베딩 모델을 준비하세요. (실행만 하면 됩니다. 모델은 처음 한 번만 내려받습니다.)

In [ ]:
# [제공 코드] 이 단원 실습에 쓸 라이브러리와 한글 폰트를 준비합니다.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer
from umap import UMAP
from sklearn.metrics.pairwise import cosine_similarity

import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지

In [ ]:
# [제공 코드] 한국어 문장 임베딩 모델을 불러옵니다(문장->768차원 벡터).
emb_model = SentenceTransformer('jhgan/ko-sroberta-multitask')

## 데이터 살펴보기

이번 과제는 고객센터에 접수된 **문의 20건**(`data/inquiries.csv`)을 사용합니다. 열은 두 개입니다 — `text`(문의 내용), `topic`(사람이 미리 달아 둔 분류: 배송·환불·품질·문의). 본격적으로 풀기 전에 데이터를 먼저 눈으로 살펴보세요(실행만 하면 됩니다).

In [ ]:
# [제공 코드] 데이터를 불러와 앞부분과 분류 분포를 살펴봅니다.
inquiries = pd.read_csv('data/inquiries.csv')
print('문의 수:', len(inquiries))
print('분류(topic) 분포:')
print(inquiries['topic'].value_counts())
display(inquiries.head())

## 1. 문의 자동 군집·토픽 파이프라인
**배경**: 고객센터에 쌓인 문의를 사람이 일일이 분류하기는 벅찹니다. 정답(분류)을 **주지 않고** 임베딩만으로 비슷한 문의끼리 **자동으로 묶고**, 각 묶음의 **대표 키워드**로 "이 묶음은 무슨 주제인가"에 **이름표(토픽 라벨)** 를 다는 작은 파이프라인을 완성합니다.

아래 각 `### N단계` 셀의 지시대로 **임베딩 → 최적 k 선택 → 군집 → 대표 키워드 → 시각화** 순서로 이어서 풉니다.

**최종 목표(자가채점 기준)**
| 단계 | 확인 항목 |
| --- | --- |
| 1단계 | `docs` 20개, `emb` shape `(20, 768)`, `coords` shape `(20, 2)` |
| 2단계 | `coords` 로 k=2~6 실루엣 비교 → 최적 군집 수 `best_k` = **2** |
| 3단계 | `best_k` 로 군집한 `labels`(길이 20), 군집 수 = `best_k` |
| 4단계 | 군집별 대표 키워드 `cluster_keywords`(군집당 6개), 한 군집에 `'결제한'` 포함 |
| 5단계 | UMAP 2D 군집 산점도(완성 그래프처럼) |
| 인사이트 | 실제 분류는 4개인데 왜 그보다 적게 묶였는지 서술 |

### 1단계 — 문의 임베딩 만들기
`data/inquiries.csv` 를 `df` 로 불러오고, 문의 본문 열(`text`)을 리스트로 만들어 `docs` 에 담으세요. 그리고 `emb_model.encode(docs)` 로 임베딩 배열 `emb` 를 만든 뒤, 교안에서처럼 **UMAP으로 2차원 좌표 `coords`** 도 만들어 둡니다(군집은 이 `coords` 위에서 합니다 — 768차원 원본은 노이즈가 커서 실루엣이 잘 안 통하기 때문이에요).

- **요구 변수**: `df`(원본), `docs`(문의 문자열 리스트, 20개), `emb`(넘파이 배열 `(20, 768)`), `coords`(넘파이 배열 `(20, 2)`).
- **주의**: `topic` 열은 **군집(2·3단계)에는 쓰지 않습니다** — 정답을 주지 않고 묶는 것이 목적이에요. 마지막 인사이트에서 결과를 대조할 때만 참고합니다.

<details><summary>힌트</summary>

```text
접근방법:
- CSV 를 데이터프레임으로 읽고, 본문 열을 리스트로 바꿔 docs 에 담은 뒤 임베딩 모델로 벡터화하고, UMAP 으로 2차원 좌표를 만든다.

세부구현:
1. read_csv 로 df 를 만든다
2. df 의 text 열을 리스트로 바꿔 docs 에 담는다
3. emb_model.encode(docs) 로 임베딩 emb 를 만든다
4. UMAP(n_components=2, n_neighbors=5, min_dist=0.05, random_state=0) 로 emb 를 2차원으로 줄여 coords 에 담는다
5. len(docs)·emb.shape·coords.shape 를 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(docs) == 20
assert emb.shape == (20, 768)
assert coords.shape == (20, 2)
print("✅ 1단계 통과!")

### 2단계 — 실루엣으로 최적 군집 수 정하기
군집을 몇 개로 나눌지 데이터가 정하게 합니다. `k` 를 **2부터 6까지** 바꿔 가며 `KMeans(n_clusters=k, random_state=0, n_init=10)` 로 **`coords`**(1단계의 2차원 좌표)를 군집화하고, 각 `k` 의 **실루엣 점수**(`silhouette_score`)를 재어 **가장 높은 `k`** 를 `best_k` 에 담으세요.

- **요구 변수**: `best_k`(실루엣이 가장 높은 군집 수).
- **요구사항**: 이 데이터에서는 `best_k` 가 **2** 로 나옵니다(k=2 의 실루엣이 약 0.59 로 가장 높음). 실제 분류는 4개(배송·환불·품질·문의)인데 지표는 2를 고릅니다 — **지표가 고른 답과 실제 분류가 다르다**는 것이 이 문제의 핵심입니다. 왜 그런지는 마지막 인사이트에서 생각해 봅니다.
- **주의**: `silhouette_score(coords, 군집라벨)` 은 좌표와 그 군집 라벨을 함께 넘깁니다. `KMeans` 는 `random_state=0, n_init=10` 으로 고정해야 결과가 재현됩니다.

<details><summary>힌트</summary>

```text
접근방법:
- k 를 2~6 으로 바꿔 가며 coords 를 군집화하고 실루엣 점수를 기록한 뒤, 점수가 가장 높은 k 를 고른다.

세부구현:
1. 점수를 담을 딕셔너리를 만든다
2. k 를 2부터 6까지 반복하며 KMeans 로 coords 를 군집화한다
3. silhouette_score 로 점수를 재어 k 별로 저장하고 출력한다
4. 점수가 가장 큰 k 를 best_k 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert best_k == 2
print("✅ 2단계 통과!")

### 3단계 — 최적 군집 수로 군집화
2단계에서 정한 `best_k` 로 `coords` 를 군집화하고, 각 문의의 군집 번호를 `labels` 에 담으세요.

- `KMeans` 로 `coords` 를 `best_k` 개 군집으로 나눠, 각 문의의 군집 번호를 `labels` 에 담습니다.
- **요구 변수**: `labels`(길이 20, 값은 0 ~ `best_k`-1).
- **주의**: 반드시 `best_k` 를 써서 2단계 선택과 일치시키고, `random_state=0, n_init=10` 을 고정하세요. 군집 번호(0·1·2) 자체엔 순서·의미가 없습니다 — 어떤 주제인지는 4단계 대표 키워드로 해석합니다.

<details><summary>힌트</summary>

```text
접근방법:
- best_k 개의 군집으로 coords 를 군집화하고 군집 라벨을 담는다.

세부구현:
1. KMeans 에 n_clusters=best_k, random_state=0, n_init=10 을 주어 coords 를 fit_predict 한다
2. 결과를 labels 에 담는다
3. 군집별 크기를 출력해 확인한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(labels) == 20
assert len(set(labels)) == best_k
print("✅ 3단계 통과!")

### 4단계 — 군집별 대표 키워드로 토픽 라벨 붙이기
군집 번호만으로는 무슨 주제인지 알 수 없습니다. 각 군집의 문의를 하나로 모아 **TF-IDF 점수가 높은 대표 단어**를 뽑아 "토픽 라벨"을 만드세요.

1. 군집마다 그 군집에 속한 문의(`df['text']` 에서 `labels == i` 인 행)를 **한 문서로 이어 붙입니다**(군집당 문자열 1개, 총 `best_k` 개 → `cluster_docs`).
2. `TfidfVectorizer(token_pattern=r'(?u)\b\w+\b')` 로 벡터화합니다 (한글 한 글자 단어도 살리려면 이 `token_pattern` 이 필요합니다).
3. 각 군집에서 TF-IDF 점수 **상위 6개 단어**를 뽑아 리스트로 만들어 `cluster_keywords`(군집당 6개 단어 리스트)에 담습니다.
4. 각 군집의 상위 단어를 이어 붙인 문자열을 `topic_labels`(군집당 1개)로 만들어 표로 출력합니다.

- **요구 변수**: `cluster_keywords`(길이 `best_k`, 각 원소는 단어 6개 리스트).
- **요구사항(자가채점)**: `best_k` 개 군집의 대표 키워드 중 **어느 한 군집**의 6개 안에 `'결제한'` 이 들어갑니다 (결제·환불처럼 주문 절차를 묻는 문의가 모인 군집). 어느 군집 번호인지는 실행마다 다를 수 있으니 **번호로 찾지 말고 전체에서** 확인하세요.
- **주의**: 군집 하나의 TF-IDF 점수 배열은 **1차원 배열로 펴서** 다뤄야 정렬이 됩니다(희소 행렬의 한 행을 그대로 쓰면 모양이 맞지 않습니다).

> **미리 알려 둡니다 — 라벨이 지저분하게 나옵니다.** '어떻게'·'있어요' 같은 조사·흔한 말이 상위에 섞여 나올 것입니다. 잘못 푼 게 아닙니다 — 교안 4절에서 본 대로 **문서가 몇 개뿐이면 IDF 가 단어를 구별하지 못하기** 때문입니다. 여기서는 '대표 키워드를 뽑는 절차'를 익히는 것이 목적이고, 제대로 된 토픽 라벨은 다음 단원(임베딩 기반 토픽 모델링)에서 만듭니다.

<details><summary>힌트</summary>

```text
접근방법:
- 군집마다 문의를 한 문서로 합쳐 TF-IDF 로 벡터화하고, 군집별 상위 6개 단어를 뽑아 라벨로 삼는다.

세부구현:
1. cluster_docs 로 군집별 합친 문서를 만든다
2. TfidfVectorizer 로 fit_transform 하고 get_feature_names_out 으로 단어 목록을 얻는다
3. 군집마다 점수 배열을 argsort 로 정렬해 상위 6개 단어를 cluster_keywords 에 담는다
4. 상위 단어를 이어 붙여 topic_labels 를 만들고 표로 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(cluster_keywords) == best_k
assert isinstance(cluster_keywords, list), '리스트로 만드세요 (LV2 의 딕셔너리와 형태가 다릅니다)'
assert all(len(words) == 6 for words in cluster_keywords)
assert any("결제한" in words for words in cluster_keywords)
print("✅ 4단계 통과!")

### 5단계 — UMAP 2D 군집 산점도
1단계에서 만든 2차원 좌표 `coords`(군집도 이 위에서 했습니다)를 군집 색으로 칠한 산점도로 그리세요. 정답 없이 묶은 군집이 공간에서 어떻게 나뉘는지 눈으로 확인합니다.

1. 새 그림을 연 뒤 `coords` 의 **첫 열을 x, 둘째 열을 y** 로 산점도를 그리되, 점 색(`c`)을 `labels` 로 주어 군집별로 다른 색이 되게 하고 제목·축 이름을 답니다.
2. **이 그래프는 자가채점이 없습니다** — 아래 완성 그래프처럼 그리면 됩니다.

<details><summary>힌트</summary>

```text
접근방법:
- 군집에 쓴 2차원 좌표 coords 를 군집 라벨을 색으로 하여 산점도로 그린다.

세부구현:
1. plt.figure 로 새 그림을 연다
2. scatter 에 coords 의 x·y 와 c=labels 를 주어 군집 색으로 그린다
3. 제목·축 이름을 달고 보여 준다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="images/과제/lv3_q1_scatter.png" width="560"/>

In [ ]:
# 여기에 코드를 작성하세요

### 인사이트 — 실제 분류는 4개인데 왜 그보다 적게 묶였을까 (서술)
실제 `topic` 은 4개(배송·환불·품질·문의)인데 실루엣은 `best_k=2` 를 골랐습니다. 군집 크기·대표 키워드(4단계)·산점도(5단계)를 근거로, **왜 지표가 고른 답과 실제 분류가 다른지** 를 **2~3문장**으로 서술하세요. (힌트: `pd.crosstab(labels, df['topic'])` 로 군집과 실제 분류를 대조해 보세요.)

> 이 단계는 자가채점이 없습니다. 정답 노트북의 모범 서술과 비교해 보세요.

*(여기에 왜 실제 분류(4개)보다 적게 묶였는지 2~3문장으로 서술하세요)*

## 2. 문의 의미 검색 도우미
**배경**: 새 문의가 들어오면, 과거 문의 중 **의미가 가장 비슷한 것**을 찾아 주는 검색 도우미를 만듭니다. 나아가 **비슷한 문의들의 분류(`topic`)를 다수결**로 모아, 새 문의를 **자동 분류**까지 해 봅니다 — 검색이 곧 분류가 되는 셈입니다.

아래 각 `### N단계` 셀의 지시대로 **사전 색인 → 질문 임베딩 → top-k 검색 함수 → 여러 질문 검색 → 다수결 분류** 순서로 이어서 풉니다.

**최종 목표(자가채점 기준)**
| 단계 | 확인 항목 |
| --- | --- |
| 1단계 | 전체 문의 사전 색인 `index_emb` shape `(20, 768)` |
| 2단계 | `"환불이 얼마나 걸려요?"` 의 가장 비슷한 문의 인덱스 = **5** |
| 3단계 | `search(query, top_k=3)` 함수 — `"환불이 얼마나 걸려요?"` 의 top1 인덱스 = 5 |
| 4단계 | 4개 질문으로 검색 결과 출력 |
| 5단계 | top-3 다수결 분류 `classify` — 4개 질문 모두 정답 분류 |
| 인사이트 | 검색이 곧 분류가 되는 원리 서술 |

### 1단계 — 전체 문의 사전 색인 만들기
다시 `data/inquiries.csv` 를 `search_df` 로 불러오고, 문의 본문(`text`) 전체를 임베딩해 **검색용 색인** `index_emb` 에 담으세요(미리 벡터로 만들어 두면 질문이 올 때마다 빠르게 비교할 수 있습니다).

- **요구 변수**: `search_df`(원본 20행), `index_emb`(넘파이 배열 `(20, 768)`).
- **주의**: 문제 1과 독립적으로 풀 수 있게 데이터를 **다시 불러옵니다**. `topic` 열은 5단계 분류에서 씁니다.

<details><summary>힌트</summary>

```text
접근방법:
- CSV 를 다시 불러와 본문 전체를 임베딩해 검색용 색인 배열로 둔다.

세부구현:
1. read_csv 로 search_df 를 만든다
2. search_df 의 text 열을 리스트로 바꿔 emb_model.encode 에 넘긴다
3. 결과를 index_emb 에 담고 shape 를 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert index_emb.shape == (20, 768)
print("✅ 1단계 통과!")

### 2단계 — 질문 임베딩과 코사인 유사도
질문 한 개를 임베딩해, 색인의 20개 문의와 **코사인 유사도**로 비교하고 **가장 비슷한 문의**를 찾으세요.

1. `question = "환불이 얼마나 걸려요?"` 를 `emb_model.encode([question])` 로 임베딩합니다(리스트로 감싸세요).
2. 질문 임베딩과 `index_emb` 의 **코사인 유사도**로 20개짜리 유사도 배열 `sims` 를 얻습니다(결과의 첫 행).
3. `sims.argmax()` 로 가장 비슷한 문의 인덱스 `top1` 을 찾아, 그 문의 내용과 분류를 출력합니다.

- **요구 변수**: `top1`(가장 비슷한 문의의 인덱스).
- **요구사항**: `"환불이 얼마나 걸려요?"` 의 `top1` 은 **5번** 문의("단순 변심으로 반품하려는데 환불은 며칠 걸리나요")입니다.
- **주의**: `encode` 에는 문장을 **리스트로** 넘겨야 2차원 배열이 나와 `cosine_similarity` 에 바로 들어갑니다.

<details><summary>힌트</summary>

```text
접근방법:
- 질문을 임베딩해 색인 전체와 코사인 유사도를 재고, 가장 큰 값의 인덱스를 찾는다.

세부구현:
1. 질문을 리스트로 감싸 encode 한다
2. cosine_similarity 로 질문과 index_emb 의 유사도 배열 sims 를 얻는다([0] 로 첫 행)
3. sims.argmax() 로 top1 을 구해 그 문의 내용·분류를 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert top1 == 5
print("✅ 2단계 통과!")

### 3단계 — top-k 검색 함수 만들기
2단계 흐름을 **함수**로 묶으세요. 질문 문자열과 개수 `top_k` 를 받아, 유사도가 높은 순서로 **상위 `top_k` 개** 문의의 정보를 돌려주는 `search(query, top_k=3)` 를 정의합니다.

- **요구 변수/함수**: `search(query, top_k=3)` — 각 결과가 `(인덱스, 유사도, 문의text, 분류topic)` 인 **리스트**를 반환(길이 `top_k`, 유사도 내림차순).
- **요구사항**: `search("환불이 얼마나 걸려요?")[0][0]` (top1 인덱스)가 **5** 여야 합니다.
- **주의**: `sims` 를 **내림차순 정렬**해 상위 `top_k` 개 인덱스를 얻습니다(`argsort` 는 오름차순이라 뒤집어야 함). `search_df` 와 `index_emb` 는 함수 밖 변수를 그대로 씁니다.

<details><summary>힌트</summary>

```text
접근방법:
- 2단계 로직(질문 임베딩→유사도→상위 인덱스)을 함수로 감싸고, 인덱스마다 정보를 튜플로 모아 반환한다.

세부구현:
1. def search(query, top_k=3): 로 함수를 연다
2. 질문을 리스트로 encode 하고 cosine_similarity 로 유사도 배열을 얻는다
3. argsort()[::-1][:top_k] 로 상위 인덱스를 고른다
4. 각 인덱스의 (인덱스, 유사도, 문의, 분류) 튜플을 리스트로 모아 반환한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(search("환불이 얼마나 걸려요?", top_k=3)) == 3
assert search("환불이 얼마나 걸려요?")[0][0] == 5
print("✅ 3단계 통과!")

### 4단계 — 여러 질문으로 검색해 보기
아래 **4개 질문**을 `test_questions` 리스트로 만들고, 각 질문마다 `search(...)` 로 가장 비슷한 문의 **top-3** 를 찾아 보기 좋게 출력하세요.

```
환불이 얼마나 걸려요?
물건이 안 왔어요
제품이 고장났어요
포인트 언제 써요?
```

- **요구 변수**: `test_questions`(위 4개 질문 리스트).
- **요구사항**: 질문마다 질문 문장과 검색된 top-3 문의(인덱스·유사도·분류·내용)를 출력합니다. 이 단계는 결과를 눈으로 확인하는 단계라 자가채점이 없습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 질문 리스트를 훑으며 각 질문에 대해 search 결과를 줄 맞춰 출력한다.

세부구현:
1. test_questions 에 질문 4개를 담는다
2. for 로 각 질문을 돌며 질문을 먼저 출력한다
3. search(질문) 결과를 하나씩 인덱스·유사도·분류·내용으로 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

### 5단계 — top-3 다수결로 자동 분류
검색된 **top-3 문의의 분류(`topic`)를 다수결**로 모아, 새 질문을 자동 분류하는 `classify(query)` 함수를 만드세요. 그리고 4단계의 4개 질문을 모두 분류해 봅니다.

1. `search(query, top_k=3)` 결과에서 각 문의의 `topic`(튜플의 4번째 값)만 모읍니다.
2. 그 분류들 중 **가장 많이 나온 것 하나**를 돌려줍니다(맨 위 제공 셀에서 `Counter` 를 이미 import 해 두었습니다 — 최빈값을 세는 데 씁니다).

- **요구 변수/함수**: `classify(query)` — top-3 문의의 다수결 분류(문자열)를 반환.
- **요구사항(자가채점)**: 4개 질문의 분류 결과가 아래와 같아야 합니다.

| 질문 | 분류 결과 |
| --- | --- |
| 환불이 얼마나 걸려요? | 환불 |
| 물건이 안 왔어요 | 배송 |
| 제품이 고장났어요 | 품질 |
| 포인트 언제 써요? | 문의 |

- **주의**: `Counter` 는 `from collections import Counter` 로 이미 준비돼 있습니다(제공 셀). top-3 을 쓰면 위 4개 질문은 동점 없이 한 분류로 정해집니다.

<details><summary>힌트</summary>

```text
접근방법:
- search 로 top-3 을 얻어 그 분류만 모으고, 가장 많이 나온 분류를 반환한다.

세부구현:
1. def classify(query): 로 함수를 연다
2. search(query, top_k=3) 결과에서 각 튜플의 topic(4번째)만 리스트로 모은다
3. Counter 로 최빈 분류 하나를 골라 반환한다 (most_common 이 (값, 횟수) 쌍의 리스트를 준다)
4. test_questions 를 돌며 질문과 분류 결과를 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert classify("환불이 얼마나 걸려요?") == "환불"
assert classify("물건이 안 왔어요") == "배송"
assert classify("제품이 고장났어요") == "품질"
assert classify("포인트 언제 써요?") == "문의"
print("✅ 5단계 통과!")

### 인사이트 — 검색이 곧 분류가 되는 원리 (서술)
따로 분류 모델을 학습하지 않았는데도 4개 질문이 모두 올바른 분류로 나뉘었습니다. **임베딩·코사인 유사도·다수결** 세 가지가 어떻게 맞물려 이런 자동 분류가 가능했는지 **2~3문장**으로 서술하세요.

> 이 단계는 자가채점이 없습니다. 정답 노트북의 모범 서술과 비교해 보세요.

*(여기에 검색이 곧 분류가 되는 원리를 2~3문장으로 서술하세요)*

---
## 3. 임베딩 모델 고르기 — 근거를 만들어 결정하기

지금까지는 `jhgan/ko-sroberta-multitask` 를 **주는 대로** 썼습니다. 하지만 실무에서 "어떤 임베딩 모델을 쓸까"는 **여러분이 결정해야 하는 문제**이고, 그 결정에는 **근거**가 필요합니다.

이 문제에서는 후보 모델 하나를 더 불러와, 문제 2에서 만든 **검색 파이프라인 그대로** 두 모델을 비교하고 **상황에 맞는 선택**을 내립니다.

| 후보 | 성격 |
|---|---|
| `jhgan/ko-sroberta-multitask` | 한국어 전용 (지금까지 쓴 모델) |
| `intfloat/multilingual-e5-small` | 다국어·경량 |

> 두 번째 모델은 처음 한 번만 내려받습니다(약 500MB, 잠시 걸려요).

In [ ]:
# [제공 코드] 두 번째 후보 모델을 불러옵니다(처음 한 번만 내려받습니다).
model_b = SentenceTransformer('intfloat/multilingual-e5-small')
print('두 번째 모델 준비 완료')

### 1단계 — 두 모델의 스펙을 표로 비교
교안 6절의 체크리스트 중 **코드로 바로 확인할 수 있는 두 가지**(임베딩 차원 · 최대 입력 길이)를 두 모델에서 각각 읽어 표로 만드세요.

- **요구 변수**: `specs` — 딕셔너리. 키는 `'ko'`(지금까지 쓴 모델)와 `'e5'`(새 모델), 값은 `{'dim': 차원, 'max_len': 최대 토큰 수}` 형태의 딕셔너리.
- 만든 `specs` 를 보기 좋은 표(`DataFrame`)로도 출력하세요.
- 두 값이 **정반대 방향으로** 다릅니다 — 어느 쪽이 무엇에 유리한지 생각하며 보세요.

<details><summary>힌트</summary>

```text
접근방법:
- 두 모델 객체 각각에서 차원과 최대 입력 길이를 읽어 딕셔너리에 담는다.

세부구현:
1. 차원은 모델의 '문장 임베딩 차원을 돌려주는 메서드'로, 최대 길이는 모델의 속성 하나로 얻는다
   (교안 6절 '문법' 블록에 두 이름이 모두 나와 있다)
2. specs = {'ko': {...}, 'e5': {...}} 형태로 담는다
3. pd.DataFrame(specs) 로 표를 만들어 display 한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert set(specs.keys()) == {'ko', 'e5'}
assert specs['ko']['dim'] == 768 and specs['ko']['max_len'] == 128
assert specs['e5']['dim'] == 384 and specs['e5']['max_len'] == 512
print("✅ 3-1단계 통과! ko 는 차원이 2배(표현력↑ 비용↑), e5 는 최대 길이가 4배(긴 글에 유리)")

### 2단계 — 같은 질문으로 두 모델의 검색 정확도 비교
스펙만으로는 고를 수 없습니다. **내 데이터로 직접 재보는 것**이 가장 확실합니다(교안 6절 퀴즈에서 본 그대로입니다).

아래 4개 질문을 두 모델로 각각 검색해, **top-1 로 뽑힌 문의의 `topic` 이 기대 분류와 맞는지** 세세요.

```python
eval_questions = [
    ('환불이 얼마나 걸려요?', '환불'),
    ('택배가 언제 오나요?', '배송'),
    ('제품에 흠집이 있어요', '품질'),
    ('회원 탈퇴하고 싶어요', '문의'),
]
```

- **요구 변수**: `acc` — 딕셔너리. 키 `'ko'`·`'e5'`, 값은 **맞힌 개수(0~4 정수)**.
- 두 모델 모두 **문의 20건 전체를 그 모델로 임베딩**한 뒤 질문과 코사인 유사도를 재야 합니다 (모델이 다르면 좌표계도 다르므로 **문서도 같은 모델로 다시 임베딩**해야 합니다 — 교안 4절 퀴즈).
- 어느 질문을 틀렸는지도 함께 출력하세요.

<details><summary>힌트</summary>

```text
접근방법:
- 모델마다 (문서 임베딩 -> 질문 임베딩 -> 코사인 -> top1 의 topic) 을 반복해 맞힌 개수를 센다.

세부구현:
1. 모델 두 개를 {'ko': model, 'e5': model_b} 처럼 딕셔너리로 묶어 반복한다
2. 모델마다 inquiries['text'] 전체를 인코딩한다 (문서도 같은 모델로 다시!)
3. 질문을 인코딩해 코사인 유사도를 구하고, 가장 큰 값의 행 번호를 찾는다
4. 그 행의 topic 이 기대 분류와 같으면 1점. 4문항을 세어 acc 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert set(acc.keys()) == {'ko', 'e5'}
assert acc['ko'] == 4, 'ko-sroberta 는 4문항을 모두 맞혀야 합니다'
assert acc['e5'] == 3, "e5 는 3문항만 맞습니다 - 3단계에서 그 이유를 밝힙니다"
print("✅ 3-2단계 통과! 스펙이 좋다고 내 데이터에서도 좋은 것은 아니다")

### 3단계 — 모델 카드가 시키는 대로 써 보기
e5 가 진 이유가 "모델이 나빠서"일까요? 아닙니다. **`e5` 계열은 입력 앞에 정해진 접두어를 붙여야** 제 성능이 납니다 — 모델 카드에 그렇게 적혀 있습니다.

- 검색할 **문서**에는 `'passage: '` 를, **질문**에는 `'query: '` 를 앞에 붙입니다.
- 2단계와 **똑같은 방식으로** e5 정확도를 다시 재어 `acc_e5_fixed` 에 담으세요(0~4 정수).
- 이것이 **모델 카드를 읽어야 하는 이유**입니다. 스펙 숫자만 보고 고르면 이런 사용 규칙을 놓칩니다.

<details><summary>힌트</summary>

```text
접근방법:
- 2단계의 e5 코드에서 문서와 질문 문자열 앞에 각각 정해진 접두어만 붙여 다시 잰다.

세부구현:
1. 문서 리스트를 만들 때 각 문의 앞에 'passage: ' 를 붙인다
2. 질문을 인코딩할 때 앞에 'query: ' 를 붙인다
3. 나머지(코사인 -> argmax -> topic 비교)는 2단계와 똑같다
4. 맞힌 개수를 acc_e5_fixed 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert acc_e5_fixed == 4, '접두어를 붙이면 4문항을 모두 맞힙니다'
assert acc_e5_fixed > acc['e5']
print("✅ 3-3단계 통과! 모델은 '정해진 사용법'대로 써야 제 성능이 난다")

### 인사이트 — 그래서 어느 모델을 쓸 것인가 (서술)
두 모델은 이제 **정확도가 같습니다(4/4)**. 그렇다면 남는 것은 스펙의 차이입니다.

| | `ko-sroberta` | `e5-small` |
|---|---|---|
| 차원 | 768 | **384** (저장·검색 비용 절반) |
| 최대 입력 | 128 토큰 | **512 토큰** (긴 글 가능) |
| 접두어 규칙 | 없음 | 있음 (`query:`/`passage:`) |

아래 **두 상황**에 각각 어느 모델을 고를지, **이유와 함께 2~3문장**으로 서술하세요.

1. **지금 이 과제처럼** 짧은 한국어 문의 수만 건을 검색하는 서비스
2. **긴 문서**(수천 자짜리 계약서·매뉴얼)를 통째로 검색해야 하는 서비스

> 이 단계는 자가채점이 없습니다. 정답 노트북의 모범 서술과 비교해 보세요.

*(여기에 두 상황별로 어느 모델을 왜 고를지 서술하세요)*